In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/datathon-2026-round-1/products.csv
/kaggle/input/competitions/datathon-2026-round-1/sample_submission.csv
/kaggle/input/competitions/datathon-2026-round-1/promotions.csv
/kaggle/input/competitions/datathon-2026-round-1/shipments.csv
/kaggle/input/competitions/datathon-2026-round-1/order_items.csv
/kaggle/input/competitions/datathon-2026-round-1/reviews.csv
/kaggle/input/competitions/datathon-2026-round-1/inventory.csv
/kaggle/input/competitions/datathon-2026-round-1/returns.csv
/kaggle/input/competitions/datathon-2026-round-1/sales.csv
/kaggle/input/competitions/datathon-2026-round-1/orders.csv
/kaggle/input/competitions/datathon-2026-round-1/geography.csv
/kaggle/input/competitions/datathon-2026-round-1/customers.csv
/kaggle/input/competitions/datathon-2026-round-1/baseline.ipynb
/kaggle/input/competitions/datathon-2026-round-1/payments.csv
/kaggle/input/competitions/datathon-2026-round-1/web_traffic.csv


**I. Data cleaning**


1.1 Bảng Master

In [3]:
df_products = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/products.csv')
check_price = (df_products['cogs'] < df_products['price']).all()
print(check_price)
df_products.info()

True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB


In [4]:
df_customers = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/customers.csv')
df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'])
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   customer_id          121930 non-null  int64         
 1   zip                  121930 non-null  int64         
 2   city                 121930 non-null  object        
 3   signup_date          121930 non-null  datetime64[ns]
 4   gender               121930 non-null  object        
 5   age_group            121930 non-null  object        
 6   acquisition_channel  121930 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 6.5+ MB


In [5]:
df_promotions = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/promotions.csv')
df_promotions['start_date'] = pd.to_datetime(df_promotions['start_date'])
df_promotions['end_date'] = pd.to_datetime(df_promotions['end_date'])
df_promotions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     object        
 1   promo_name           50 non-null     object        
 2   promo_type           50 non-null     object        
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[ns]
 5   end_date             50 non-null     datetime64[ns]
 6   applicable_category  10 non-null     object        
 7   promo_channel        50 non-null     object        
 8   stackable_flag       50 non-null     int64         
 9   min_order_value      50 non-null     int64         
dtypes: datetime64[ns](2), float64(1), int64(2), object(5)
memory usage: 4.0+ KB


In [6]:
df_geography = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/geography.csv')
df_geography.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   zip       39948 non-null  int64 
 1   city      39948 non-null  object
 2   region    39948 non-null  object
 3   district  39948 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.2+ MB


1.2 Bảng Transaction

In [7]:
df_orders = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/orders.csv')
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  int64         
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  int64         
 3   zip             646945 non-null  int64         
 4   order_status    646945 non-null  object        
 5   payment_method  646945 non-null  object        
 6   device_type     646945 non-null  object        
 7   order_source    646945 non-null  object        
dtypes: datetime64[ns](1), int64(3), object(4)
memory usage: 39.5+ MB


In [8]:
df_order_items = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/order_items.csv')
df_order_items.info()
df_order_items['promo_id_2'].unique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB


/tmp/ipykernel_55/3428759194.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_order_items = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/order_items.csv')


array([nan, 'PROMO-0015', 'PROMO-0025'], dtype=object)

In [9]:
df_payments = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/payments.csv')
df_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   payment_method  646945 non-null  object 
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 19.7+ MB


In [10]:
df_shipments = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/shipments.csv')
df_shipments['ship_date'] = pd.to_datetime(df_shipments['ship_date'])
df_shipments['delivery_date'] = pd.to_datetime(df_shipments['delivery_date'])
df_shipments.info()
df_shipments.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   order_id       566067 non-null  int64         
 1   ship_date      566067 non-null  datetime64[ns]
 2   delivery_date  566067 non-null  datetime64[ns]
 3   shipping_fee   566067 non-null  float64       
dtypes: datetime64[ns](2), float64(1), int64(1)
memory usage: 17.3 MB


,order_id,ship_date,delivery_date,shipping_fee
0,1,2012-07-07,2012-07-11,1.37
1,2,2012-07-06,2012-07-10,2.60
2,3,2012-07-04,2012-07-07,2.38
3,4,2012-07-05,2012-07-11,2.49
4,6,2012-07-09,2012-07-16,25.79
5,7,2012-07-06,2012-07-12,1.31
6,8,2012-07-06,2012-07-11,0.43
7,9,2012-07-07,2012-07-13,0.18
8,10,2012-07-07,2012-07-13,0.86
9,13,2012-07-09,2012-07-12,2.44


In [11]:
df_returns = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/returns.csv')
df_returns['return_date'] = pd.to_datetime(df_returns['return_date'])
df_returns.info()
df_returns.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   return_id        39939 non-null  object        
 1   order_id         39939 non-null  int64         
 2   product_id       39939 non-null  int64         
 3   return_date      39939 non-null  datetime64[ns]
 4   return_reason    39939 non-null  object        
 5   return_quantity  39939 non-null  int64         
 6   refund_amount    39939 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 2.1+ MB


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76
5,RET-000006,59,671,2012-07-19,defective,1,10086.33
6,RET-000007,67,604,2012-07-16,wrong_size,1,5713.22
7,RET-000008,102,467,2012-07-17,defective,1,9724.09
8,RET-000010,108,635,2012-07-30,wrong_size,5,43387.54
9,RET-000012,132,103,2012-07-29,changed_mind,2,19200.35


In [12]:
df_reviews = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/reviews.csv')
df_reviews['review_date'] = pd.to_datetime(df_reviews['review_date'])
df_reviews.info()
df_reviews.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113551 entries, 0 to 113550
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   review_id     113551 non-null  object        
 1   order_id      113551 non-null  int64         
 2   product_id    113551 non-null  int64         
 3   customer_id   113551 non-null  int64         
 4   review_date   113551 non-null  datetime64[ns]
 5   rating        113551 non-null  int64         
 6   review_title  113551 non-null  object        
dtypes: datetime64[ns](1), int64(4), object(2)
memory usage: 6.1+ MB


,review_id,order_id,product_id,customer_id,review_date,rating,review_title
0,REV-0000001,1,2400,58578,2012-07-24,5,Highly recommend
1,REV-0000002,3,396,58811,2012-08-03,5,Very satisfied
2,REV-0000003,10,1431,49101,2012-07-23,5,Great quality
3,REV-0000005,16,1668,41028,2012-08-05,5,Great quality
4,REV-0000006,17,2352,42030,2012-07-17,4,Good overall
5,REV-0000008,20,2241,42962,2012-08-12,5,Great quality
6,REV-0000009,23,1041,12213,2012-07-19,5,Very satisfied
7,REV-0000010,24,671,13426,2012-08-09,4,Solid choice
8,REV-0000011,28,396,9473,2012-07-27,5,Very satisfied
9,REV-0000012,49,671,39696,2012-07-17,5,Highly recommend


1.3 Bảng Analytical

In [13]:
df_sales = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/sales.csv')
df_sales['Date'] = pd.to_datetime(df_sales['Date'])
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     3833 non-null   datetime64[ns]
 1   Revenue  3833 non-null   float64       
 2   COGS     3833 non-null   float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 90.0 KB


1.4 Bảng Operational

In [14]:
df_inventory = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/inventory.csv')
df_inventory['snapshot_date'] = pd.to_datetime(df_inventory['snapshot_date'])
df_inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60247 entries, 0 to 60246
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   snapshot_date      60247 non-null  datetime64[ns]
 1   product_id         60247 non-null  int64         
 2   stock_on_hand      60247 non-null  int64         
 3   units_received     60247 non-null  int64         
 4   units_sold         60247 non-null  int64         
 5   stockout_days      60247 non-null  int64         
 6   days_of_supply     60247 non-null  float64       
 7   fill_rate          60247 non-null  float64       
 8   stockout_flag      60247 non-null  int64         
 9   overstock_flag     60247 non-null  int64         
 10  reorder_flag       60247 non-null  int64         
 11  sell_through_rate  60247 non-null  float64       
 12  product_name       60247 non-null  object        
 13  category           60247 non-null  object        
 14  segmen

In [15]:
df_web_traffic = pd.read_csv('/kaggle/input/competitions/datathon-2026-round-1/web_traffic.csv')
df_web_traffic['date'] = pd.to_datetime(df_web_traffic['date'])
df_web_traffic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      3652 non-null   datetime64[ns]
 1   sessions                  3652 non-null   int64         
 2   unique_visitors           3652 non-null   int64         
 3   page_views                3652 non-null   int64         
 4   bounce_rate               3652 non-null   float64       
 5   avg_session_duration_sec  3652 non-null   float64       
 6   traffic_source            3652 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(3), object(1)
memory usage: 199.8+ KB


**Merge Table**

In [16]:
returns_agg = df_returns.groupby(["order_id", "product_id"], as_index=False).agg(
    return_quantity=("return_quantity", "sum"),
    refund_amount=("refund_amount", "sum")
)

reviews_agg = df_reviews.groupby(["order_id", "product_id", "customer_id"], as_index=False).agg(
    avg_rating=("rating", "mean")
)

order_items_v2 = df_order_items.merge(df_orders, on="order_id", how="left")
order_items_v2 = order_items_v2.merge(df_products, on="product_id", how="left")
order_items_v2 = order_items_v2.merge(df_customers, on="customer_id", how="left")
order_items_v2 = order_items_v2.merge(df_shipments, on="order_id", how="left")
order_items_v2 = order_items_v2.merge(returns_agg, on=["order_id", "product_id"], how="left")
order_items_v2 = order_items_v2.merge(reviews_agg, on=["order_id", "product_id", "customer_id"], how="left")
order_items_v2 = order_items_v2.merge(df_geography[["city", "region"]].drop_duplicates(), on="city", how="left")

In [17]:
order_items_v2["gross_line_amount"] = order_items_v2["quantity"] * order_items_v2["unit_price"]
order_items_v2["line_cogs"] = order_items_v2["quantity"] * order_items_v2["cogs"]
order_items_v2["line_margin"] = order_items_v2["gross_line_amount"] - order_items_v2["line_cogs"]
order_items_v2["delivery_days"] = (order_items_v2["delivery_date"] - order_items_v2["ship_date"]).dt.days
order_items_v2 = order_items_v2.drop(columns=['zip_x','zip_y','product_name'])

In [18]:
order_items_v2.info()
order_items_v2.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 35 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             714669 non-null  int64         
 1   product_id           714669 non-null  int64         
 2   quantity             714669 non-null  int64         
 3   unit_price           714669 non-null  float64       
 4   discount_amount      714669 non-null  float64       
 5   promo_id             276316 non-null  object        
 6   promo_id_2           206 non-null     object        
 7   order_date           714669 non-null  datetime64[ns]
 8   customer_id          714669 non-null  int64         
 9   order_status         714669 non-null  object        
 10  payment_method       714669 non-null  object        
 11  device_type          714669 non-null  object        
 12  order_source         714669 non-null  object        
 13  category      

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,customer_id,order_status,...,delivery_date,shipping_fee,return_quantity,refund_amount,avg_rating,region,gross_line_amount,line_cogs,line_margin,delivery_days
0,1,2400,7,1138.22,0.0,NaN,NaN,2012-07-04,58578,delivered,...,2012-07-11,1.37,NaN,NaN,5.0,East,7967.54,7376.586059,590.953941,4.0
1,2,609,7,10166.25,0.0,NaN,NaN,2012-07-04,58621,returned,...,2012-07-10,2.60,6.0,52458.01,NaN,East,71163.75,62913.929616,8249.820384,4.0
2,3,396,3,11220.33,0.0,NaN,NaN,2012-07-04,58811,delivered,...,2012-07-07,2.38,NaN,NaN,5.0,East,33660.99,30273.036767,3387.953233,3.0
3,4,635,5,10639.25,0.0,NaN,NaN,2012-07-04,59453,delivered,...,2012-07-11,2.49,NaN,NaN,NaN,East,53196.25,46027.152390,7169.097610,6.0
4,6,1935,1,1597.84,0.0,NaN,NaN,2012-07-06,57821,delivered,...,2012-07-16,25.79,NaN,NaN,NaN,East,1597.84,1048.696357,549.143643,7.0
5,7,1934,6,1633.49,0.0,NaN,NaN,2012-07-06,57820,delivered,...,2012-07-12,1.31,NaN,NaN,NaN,East,9800.94,8027.018786,1773.921214,6.0
6,8,1934,6,1602.92,0.0,NaN,NaN,2012-07-06,57818,delivered,...,2012-07-11,0.43,NaN,NaN,NaN,East,9617.52,8027.018786,1590.501214,5.0
7,8,1935,4,1642.51,0.0,NaN,NaN,2012-07-06,57818,delivered,...,2012-07-11,0.43,NaN,NaN,NaN,East,6570.04,4194.785429,2375.254571,5.0
8,9,1432,8,4049.64,0.0,NaN,NaN,2012-07-06,49102,delivered,...,2012-07-13,0.18,NaN,NaN,NaN,East,32397.12,31112.424000,1284.696000,6.0
9,10,1431,5,3977.37,0.0,NaN,NaN,2012-07-06,49101,delivered,...,2012-07-13,0.86,NaN,NaN,5.0,East,19886.85,13366.061100,6520.788900,6.0


In [19]:
daily_orders = order_items_v2.groupby("order_date", as_index=False).agg(
    order_count=("order_id", "nunique"),
    item_count=("quantity", "sum"),
    product_count=("product_id", "nunique"),
    customer_count=("customer_id", "nunique"),
    order_revenue_calc=("gross_line_amount", "sum"),
    order_cogs_calc=("line_cogs", "sum"),
    total_discount=("discount_amount", "sum"),
    total_refund=("refund_amount", "sum"),
    avg_delivery_days=("delivery_days", "mean")
).rename(columns={"order_date": "Date"})

daily_traffic = df_web_traffic.groupby("date", as_index=False).agg(
    sessions=("sessions", "sum"),
    unique_visitors=("unique_visitors", "sum"),
    page_views=("page_views", "sum"),
    bounce_rate=("bounce_rate", "mean")
).rename(columns={"date": "Date"})

daily_table = df_sales.merge(daily_orders, on="Date", how="left")
daily_table = daily_table.merge(daily_traffic, on="Date", how="left")

daily_table["day_of_week"] = daily_table["Date"].dt.dayofweek
daily_table["month"] = daily_table["Date"].dt.month
daily_table["year"] = daily_table["Date"].dt.year
daily_table["is_weekend"] = np.where(daily_table["day_of_week"].isin([5, 6]), 1, 0)



In [20]:
daily_table.head(10)

,Date,Revenue,COGS,order_count,item_count,product_count,customer_count,order_revenue_calc,order_cogs_calc,total_discount,total_refund,avg_delivery_days,sessions,unique_visitors,page_views,bounce_rate,day_of_week,month,year,is_weekend
0,2012-07-04,5123547.94,3982991.19,162,777,93,161,5123547.94,3.982991e+06,0.0,171067.01,4.310127,NaN,NaN,NaN,NaN,2,7,2012,0
1,2012-07-05,2751773.45,2150580.23,97,428,65,97,2751773.45,2.150580e+06,0.0,57342.91,4.655914,NaN,NaN,NaN,NaN,3,7,2012,0
2,2012-07-06,3054029.42,2517632.84,93,441,73,93,3054029.42,2.517633e+06,0.0,195614.58,4.423529,NaN,NaN,NaN,NaN,4,7,2012,0
3,2012-07-07,2667930.94,2108246.62,73,364,50,73,2667930.94,2.108247e+06,0.0,134785.14,4.390625,NaN,NaN,NaN,NaN,5,7,2012,1
4,2012-07-08,2360851.90,1808622.79,88,394,62,87,2360851.90,1.808623e+06,0.0,135871.83,4.345238,NaN,NaN,NaN,NaN,6,7,2012,1
5,2012-07-09,3548386.46,2787841.68,137,730,92,137,3548386.46,2.787842e+06,0.0,165093.16,4.416667,NaN,NaN,NaN,NaN,0,7,2012,0
6,2012-07-10,5234938.62,4044438.84,183,928,131,182,5234938.62,4.044439e+06,0.0,112143.07,4.171598,NaN,NaN,NaN,NaN,1,7,2012,0
7,2012-07-11,5582884.78,4338313.07,221,1097,130,221,5582884.78,4.338313e+06,0.0,173779.73,4.534884,NaN,NaN,NaN,NaN,2,7,2012,0
8,2012-07-12,5734632.02,4458811.27,246,1241,157,243,5734632.02,4.458811e+06,0.0,94601.97,4.349794,NaN,NaN,NaN,NaN,3,7,2012,0
9,2012-07-13,5309511.71,4143402.78,201,995,118,200,5309511.71,4.143403e+06,0.0,56152.87,4.478947,NaN,NaN,NaN,NaN,4,7,2012,0


In [21]:
daily_table = daily_table.drop(columns=['order_revenue_calc', 'order_cogs_calc'])
daily_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               3833 non-null   datetime64[ns]
 1   Revenue            3833 non-null   float64       
 2   COGS               3833 non-null   float64       
 3   order_count        3833 non-null   int64         
 4   item_count         3833 non-null   int64         
 5   product_count      3833 non-null   int64         
 6   customer_count     3833 non-null   int64         
 7   total_discount     3833 non-null   float64       
 8   total_refund       3833 non-null   float64       
 9   avg_delivery_days  3831 non-null   float64       
 10  sessions           3652 non-null   float64       
 11  unique_visitors    3652 non-null   float64       
 12  page_views         3652 non-null   float64       
 13  bounce_rate        3652 non-null   float64       
 14  day_of_w

In [22]:
# =========================
# ADD PROMOTION FEATURES - HANDLE PERCENTAGE / FIXED
# =========================

import pandas as pd
import numpy as np

promotions = pd.read_csv("/kaggle/input/competitions/datathon-2026-round-1/promotions.csv")

# chỉ giữ cột cần thiết
promotions = promotions[
    ["promo_type", "discount_value", "start_date", "end_date"]
].copy()

promotions["start_date"] = pd.to_datetime(promotions["start_date"])
promotions["end_date"] = pd.to_datetime(promotions["end_date"])
daily_table["Date"] = pd.to_datetime(daily_table["Date"])

# fixed: 50.0 nghĩa là 50,000 VND
promotions["fixed_discount_vnd"] = np.where(
    promotions["promo_type"] == "fixed",
    promotions["discount_value"] * 1000,
    0
)

# percentage: 12.0 nghĩa là 12%
promotions["percentage_discount"] = np.where(
    promotions["promo_type"] == "percentage",
    promotions["discount_value"],
    0
)

promo_daily_list = []

for _, row in promotions.iterrows():
    dates = pd.date_range(row["start_date"], row["end_date"], freq="D")

    temp = pd.DataFrame({
        "Date": dates,
        "promo_percentage_discount": row["percentage_discount"],
        "promo_fixed_discount_vnd": row["fixed_discount_vnd"]
    })

    promo_daily_list.append(temp)

promo_daily = pd.concat(promo_daily_list, ignore_index=True)

promo_features = promo_daily.groupby("Date").agg(
    promo_count=("promo_percentage_discount", "count"),

    promo_avg_percentage_discount=("promo_percentage_discount", "mean"),
    promo_max_percentage_discount=("promo_percentage_discount", "max"),
    promo_total_percentage_discount=("promo_percentage_discount", "sum"),

    promo_avg_fixed_discount_vnd=("promo_fixed_discount_vnd", "mean"),
    promo_max_fixed_discount_vnd=("promo_fixed_discount_vnd", "max"),
    promo_total_fixed_discount_vnd=("promo_fixed_discount_vnd", "sum")
).reset_index()

# xoá cột promo cũ nếu đã merge trước đó
old_promo_cols = [
    col for col in daily_table.columns
    if col.startswith("promo_") or col == "is_promo_day"
]

daily_table = daily_table.drop(columns=old_promo_cols, errors="ignore")

# merge vào daily_table
daily_table = daily_table.merge(promo_features, on="Date", how="left")

# fill ngày không có khuyến mãi
promo_cols = [col for col in daily_table.columns if col.startswith("promo_")]
daily_table[promo_cols] = daily_table[promo_cols].fillna(0)

daily_table["is_promo_day"] = (daily_table["promo_count"] > 0).astype(int)

daily_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Date                             3833 non-null   datetime64[ns]
 1   Revenue                          3833 non-null   float64       
 2   COGS                             3833 non-null   float64       
 3   order_count                      3833 non-null   int64         
 4   item_count                       3833 non-null   int64         
 5   product_count                    3833 non-null   int64         
 6   customer_count                   3833 non-null   int64         
 7   total_discount                   3833 non-null   float64       
 8   total_refund                     3833 non-null   float64       
 9   avg_delivery_days                3831 non-null   float64       
 10  sessions                         3652 non-null   float64    

In [23]:
daily_table.head(500)

,Date,Revenue,COGS,order_count,item_count,product_count,customer_count,total_discount,total_refund,avg_delivery_days,...,year,is_weekend,promo_count,promo_avg_percentage_discount,promo_max_percentage_discount,promo_total_percentage_discount,promo_avg_fixed_discount_vnd,promo_max_fixed_discount_vnd,promo_total_fixed_discount_vnd,is_promo_day
0,2012-07-04,5123547.94,3982991.19,162,777,93,161,0.0,171067.01,4.310127,...,2012,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2012-07-05,2751773.45,2150580.23,97,428,65,97,0.0,57342.91,4.655914,...,2012,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2012-07-06,3054029.42,2517632.84,93,441,73,93,0.0,195614.58,4.423529,...,2012,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2012-07-07,2667930.94,2108246.62,73,364,50,73,0.0,134785.14,4.390625,...,2012,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2012-07-08,2360851.90,1808622.79,88,394,62,87,0.0,135871.83,4.345238,...,2012,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2013-11-11,3495969.18,2676318.51,153,815,114,152,0.0,76269.57,4.760000,...,2013,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
496,2013-11-12,5162437.56,3979066.87,224,1166,133,218,0.0,47044.32,4.393013,...,2013,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
497,2013-11-13,4681263.25,3622930.35,171,943,113,170,0.0,242699.99,4.672131,...,2013,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
498,2013-11-14,3118904.74,2460188.52,122,630,88,121,0.0,62721.65,4.720000,...,2013,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
daily_table.to_csv('daily_table.csv', index=False)